In [1]:
from torch import nn
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
import pytorch_lightning as pl
import segmentation_models_pytorch as smp
from torch import optim

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint

import os
from pathlib import Path
import tqdm
from skimage import io
import pandas as pd

d:\Git\autoSCAPE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0810 16:32:35.459000 29456 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [2]:
torch.set_float32_matmul_precision('medium')

In [3]:
torch.cuda.is_available()

True

## Define model

In [ ]:
import lightning as pl
import segmentation_models_pytorch as smp
from torch import nn, optim
import torch.nn.functional as F
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

class FocusLightningModel(pl.LightningModule):
    def __init__(self, weights="resnet18", selected_levels=[1, 2], use_pooling=True):
        super(FocusLightningModel, self).__init__()
        self.selected_levels = selected_levels
        self.use_pooling = use_pooling

        # Load the pre-trained ResNet18 model
        unet = smp.Unet(
            weights,
            encoder_weights="imagenet",
            in_channels=1,
            classes=1,
            activation=None,
        )

        # Use the convolutional part as the encoder
        self.encoder = unet.encoder
        
        # Calculate feature dimension dynamically
        # Get encoder output channels for selected levels
        # ResNet encoder outputs: [64, 64, 128, 256, 512] for levels 0-4
        encoder_channels = self.encoder.out_channels
        self.selected_channels = sum([encoder_channels[i] for i in selected_levels])
        
        # Calculate input dimension for FC layers
        if use_pooling:
            # With global average pooling, we only need channels as features
            fc_input_dim = self.selected_channels
        else:
            # Without pooling, calculate based on feature map sizes
            # For a 140x140 input to ResNet18: 
            # Level 1: 70x70, Level 2: 35x35
            feature_sizes = [70*70, 35*35]  # Adjust based on your encoder
            fc_input_dim = sum([encoder_channels[i] * feature_sizes[j] 
                               for j, i in enumerate(selected_levels)])
        
        # Add a bottleneck layer to reduce dimensionality before FC layers
        bottleneck_dim = 1024
        self.bottleneck = nn.Sequential(
            nn.Linear(fc_input_dim, bottleneck_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Fully connected layers (now much smaller!)
        self.fc1 = nn.Linear(bottleneck_dim, 256)
        self.fc2 = nn.Linear(256, 1)
        
        # Global average pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1) if use_pooling else None

        self.criterion = nn.MSELoss()
        self.log_dict = {}
        self.log_dict["weights"] = weights
        
        # Log parameter count
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Total parameters: {total_params:,}")
        print(f"FC input dimension: {fc_input_dim:,}")

    def forward(self, x):
        # Get all encoder outputs
        encoder_outputs = self.encoder(x)
        
        # Select only the levels we need
        selected_features = [encoder_outputs[i] for i in self.selected_levels]
        
        # Option 1: Use global average pooling to reduce spatial dimensions
        if self.use_pooling:
            pooled_features = [self.global_pool(f).view(f.size(0), -1) for f in selected_features]
            x = torch.cat(pooled_features, dim=1)
        else:
            # Option 2: Flatten directly (less aggressive compression)
            flattened_features = [f.view(f.size(0), -1) for f in selected_features]
            x = torch.cat(flattened_features, dim=1)
        
        # Pass through bottleneck to reduce dimensionality
        x = self.bottleneck(x)
        
        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def training_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self(images)
        loss = self.criterion(outputs.squeeze(), labels.float())
        self.log("train_loss", loss)
        return {"loss": loss}

    def validation_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self(images)
        loss = self.criterion(outputs.squeeze(), labels.float())

        if not hasattr(self, "val_outputs"):
            self.val_outputs = []
        self.val_outputs.append(
            {
                "val_loss": loss,
            }
        )
        return {"val_loss": loss}

    def on_validation_epoch_end(self):
        avg_loss = torch.stack([x["val_loss"] for x in self.val_outputs]).mean()
        self.log("val_loss", avg_loss, prog_bar=True)

        # Optionally clear the outputs to free up memory
        del self.val_outputs

    def test_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self(images)
        loss = self.criterion(outputs.squeeze(), labels.float())
        self.log("test_loss", loss)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=0.001)
        return optimizer

def preprocess_stack(stack, random: bool = True):

    if random:
        augmentations = A.Compose(
            [
                A.RandomRotate90(p=0.5),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.Transpose(p=0.5),
                A.ShiftScaleRotate(p=0.5),
                A.Normalize(mean=0, std=1, max_pixel_value=255.0),
                A.Resize(height=140, width=140),
                ToTensorV2(),
            ]
        )
    else:
        augmentations = A.Compose(
            [
                A.Normalize(mean=0, std=1, max_pixel_value=255.0),
                A.Resize(height=140, width=140),
                ToTensorV2(),
            ]
        )

    processed_stack = []
    for slice in stack:
        # If grayscale, repeat the single channel three times to simulate RGB
        processed_slice = augmentations(image=slice)["image"]
        processed_stack.append(processed_slice)

    return torch.stack(processed_stack)


def get_focus_plane(model, image_stack):
    import numpy as np

    image_stack = preprocess_stack(image_stack).to("cuda")
    with torch.no_grad():
        offsets = model(image_stack)
        offsets = offsets.squeeze().detach().cpu().numpy()

    # retrieve focus plane
    return np.argmin(abs(offsets))


def detect_focus_plane(
    image: "napari.types.ImageData",
    segmentation: "napari.types.LabelsData",
    width: float,
    height: float,
    scale_xy: float,
    crop_margin: int = 0,
    focus_detector: torch.nn.Module = None,
) -> "napari.types.PointsData":
    from .utilities import crop_well_from_image, get_well_centroids
    import tqdm
    import numpy as np

    well_centroids = get_well_centroids(segmentation) / scale_xy
    focus_planes = []
    for idx in tqdm.tqdm(range(len(well_centroids)), desc="Detecting focus planes"):
        try:
            well = crop_well_from_image(image, well_centroids[idx], width, height)
            well = well[:, crop_margin:-crop_margin, crop_margin:-crop_margin]

            focus_plane = get_focus_plane(focus_detector, well)
            focus_planes.append(focus_plane)
        except:
            focus_planes.append(np.nan)

    focus_planes = np.asarray(focus_planes)
    focus_points = np.concatenate((focus_planes[:, None], well_centroids), axis=1)

    return focus_points

In [5]:
working_directory = r'E:\UserData\Johannes\20260424\focus_detector_training'
images_dir = os.path.join(working_directory, 'images')
labels_dir = os.path.join(working_directory, 'labels')
os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

## Get training data from OMERO

In [6]:
download_again = False

if download_again:
    import ezomero

    host = 'omero-nfdi.uni-muenster.de'
    port = 4064
    user = 'JohannesSoltwedel'
    group = None
    secure=True
    password = "BYMCfyZpMDLC"
    dataset_id = 301

    conn = ezomero.connect(host=host, user=user, secure=secure, port=port, group=group, password=password)

    img_ids = ezomero.get_image_ids(conn, dataset=dataset_id)  # replace 259 with the id of the dataset you want to use
    img_ids = [i for i in tqdm.tqdm(img_ids) if len(ezomero.get_map_annotation_ids(conn, object_type="Image", object_id=i)) > 0]  # filter images that have kv-annotation

    df_annotation = pd.DataFrame(columns=["File", "z-slice"])
    for img_id in tqdm.tqdm(img_ids):
        if not os.path.exists(os.path.join(images_dir, f"{img_id}.tif")):
            image_wrapper, image_stack = ezomero.get_image(conn, img_id, dim_order="tczyx")
            filename = os.path.join(images_dir, f"{img_id}.tif")
            io.imsave(filename, image_stack.astype("uint16").squeeze(), check_contrast=False)

        annotation_id = ezomero.get_map_annotation_ids(conn, object_type="Image", object_id=img_id)[0]
        annotation = ezomero.get_map_annotation(conn, annotation_id)

        _df = pd.DataFrame([{"File": Path(filename).stem + ".tif", "z-slice": annotation["focus_plane"]}])
        df_annotation = pd.concat((df_annotation, _df), ignore_index=True)

    df_annotation.to_csv(os.path.join(labels_dir, "annotations.csv"), index=False)

## Dataset

In [7]:
class FocusDataset(Dataset):
    def __init__(self, dataframe, augment=True, preload=True):
        self.dataframe = dataframe
        self.augment = augment
        self.preload = preload
        
        # Build a flat dataset with (file_path, annotated_slice, z_slice) tuples
        # This allows us to return multiple slices from the same stack
        self.samples = []
        self.stack_cache = {}  # Cache for loaded stacks
        
        # Get unique files and their metadata
        unique_files = self.dataframe.groupby('File').first()
        
        for file_path, row in unique_files.iterrows():
            annotated_slice = row['z-slice']
            
            # Load stack once to get its shape
            stack = io.imread(file_path)
            total_slices = stack.shape[0]
            
            # Preload if requested
            if self.preload:
                self.stack_cache[file_path] = stack
            
            # Create samples for each z-slice in this stack
            # This way, we load each stack only once but get multiple samples from it
            for z_idx in range(total_slices):
                self.samples.append({
                    'file': file_path,
                    'z_idx': z_idx,
                    'annotated_z': annotated_slice,
                    'total_slices': total_slices
                })
        
        print(f"Dataset initialized with {len(self.samples)} samples from {len(unique_files)} stacks")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample_info = self.samples[idx]
        file_path = sample_info['file']
        z_idx = sample_info['z_idx']
        annotated_z = sample_info['annotated_z']
        
        # Load from cache or disk
        if file_path in self.stack_cache:
            stack = self.stack_cache[file_path]
        else:
            stack = io.imread(file_path)
            self.stack_cache[file_path] = stack  # Cache for future use
        
        # Get the specific slice
        image = stack[z_idx]
        
        # Calculate relative position to the annotated z-slice
        relative_position = abs(z_idx - annotated_z)
        
        # Apply augmentations if requested
        if self.augment:
            augmentations = A.Compose([
                A.RandomRotate90(p=0.5),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.Transpose(p=0.5),
                A.Normalize(mean=0, std=1, max_pixel_value=255.0),
                A.Resize(height=140, width=140),
                ToTensorV2()
            ])
            image = augmentations(image=image)['image']
        else:
            augmentations = A.Compose([
                A.Normalize(mean=0, std=1, max_pixel_value=255.0),
                A.Resize(height=140, width=140),
                ToTensorV2()
            ])
            image = augmentations(image=image)['image']
        
        sample = {
            'image': image,
            'label': relative_position,
            'File': file_path,
        }
        
        return sample


In [8]:
df = pd.read_csv(os.path.join(labels_dir, "annotations.csv"))

# add root to full file path
df.File = [os.path.join(working_directory, 'images', f) for f in df.File]

# Splitting the dataset into training (70%) and temporary set (30%)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)

# Splitting the temporary set into validation (2/3 of 30% = 20%) and test (1/3 of 30% = 10%)
val_df, test_df = train_test_split(temp_df, test_size=0.33, random_state=42)

# Now, you can use these DataFrames to create separate dataset instances:
train_dataset = FocusDataset(train_df)
val_dataset = FocusDataset(val_df)
test_dataset = FocusDataset(test_df)

Dataset initialized with 429 samples from 39 stacks
Dataset initialized with 121 samples from 11 stacks
Dataset initialized with 66 samples from 6 stacks


## Training

This will generate a folder for each training run next to this notebook ('logs_focus_detector'), where you'll find the checkpoint model file for further usage. For viosualixzation, install tensorboard like this:
'''bash
pip install tensorboard
'''

Then, open a new miniforge prompt, activate the environment and type

'''bash
tensorboard --logdir=path/to/log/directory
'''

and navigate to [this page](http://localhost:6006/) in your browser



In [9]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [21]:
model = FocusLightningModel()

Total parameters: 11,367,873
FC input dimension: 128


### Checkpoints and callbacks

In [22]:
# Setting up Model Checkpointing
checkpoint_callback = ModelCheckpoint(
    save_top_k=1,  # Save the best model only and call it best.pt
    save_last=True,  # Also save the last model
    verbose=True,
    monitor='val_loss',  # Assuming you have a validation loss logged with this name
    mode='min',  # Save the model with the minimum 'val_loss',
    filename='best'
)

In [23]:
trainer = pl.Trainer(
    max_epochs=200,
    default_root_dir='./logs_focus_detector',  # Directory for TensorBoard logs
    callbacks=[checkpoint_callback])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [24]:
trainer.fit(model, train_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type              | Params | Mode  | FLOPs
------------------------------------------------------------------
0 | encoder     | ResNetEncoder     | 11.2 M | train | 0    
1 | bottleneck  | Sequential        | 66.0 K | train | 0    
2 | fc1         | Linear            | 131 K  | train | 0    
3 | fc2         | Linear            | 257    | train | 0    
4 | global_pool | AdaptiveAvgPool2d | 0      | train | 0    
5 | criterion   | MSELoss           | 0      | train | 0    
------------------------------------------------------------------
11.4 M    Trainable params
0         Non-trainable params
11.4 M    Total params
45.471    Total estimated model params size (MB)
74        Modules in train mode
0         Modules in eval mod

Epoch 0: 100%|██████████| 7/7 [00:00<00:00, 15.56it/s, v_num=4, val_loss=8.290]

Epoch 0, global step 7: 'val_loss' reached 8.29323 (best 8.29323), saving model to 'logs_focus_detector\\lightning_logs\\version_4\\checkpoints\\best.ckpt' as top 1


Epoch 1: 100%|██████████| 7/7 [00:00<00:00, 15.06it/s, v_num=4, val_loss=7.910]

Epoch 1, global step 14: 'val_loss' reached 7.91179 (best 7.91179), saving model to 'logs_focus_detector\\lightning_logs\\version_4\\checkpoints\\best.ckpt' as top 1


...

Epoch 199: 100%|██████████| 7/7 [00:00<00:00, 14.11it/s, v_num=4, val_loss=3.160]

Epoch 199, global step 1400: 'val_loss' was not in top 1


Epoch 199: 100%|██████████| 7/7 [00:00<00:00, 14.02it/s, v_num=4, val_loss=3.160]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 199: 100%|██████████| 7/7 [00:00<00:00, 11.48it/s, v_num=4, val_loss=3.160]
